## Baseline (no fine-tuning) — Task 3 (Provision-Type Classification / LEDGAR)

This notebook is an **exact replica** of the data pipeline in [llm_fine_tuning_LORA_task3.ipynb](llm_fine_tuning_LORA_task3.ipynb) with **one difference: there is no training**. The base **`meta-llama/Meta-Llama-3.1-8B`** model is loaded and used *as-is* to classify the **same 1,945-row validation set** the fine-tuned run is evaluated on, so the two sets of metrics are directly comparable. It mirrors the Task 1 / Task 2 baseline notebooks ([llama_3.1_task_1_no_fine_tune.ipynb](llama_3.1_task_1_no_fine_tune.ipynb), [llama_3.1_task_2_no_fine_tune.ipynb](llama_3.1_task_2_no_fine_tune.ipynb)).

Task 3 = given one contract **provision**, classify it into **one of 100 label types** (e.g. *Governing Laws*, *Notices*, *Terminations*).

**Why this baseline matters:** fine-tuning teaches two skills at once here — picking the right label *and* answering with a bare, valid label string instead of prose. Running the **un-fine-tuned** base model through the *identical* evaluation (valid-label rate → accuracy → macro/micro-F1 → per-label → confusions) tells us what the model can do out-of-the-box, which is the reference point fine-tuning is measured against. The base model sees the **same full 100-label instruction menu**, so the comparison is fair.

The data pipeline (load LEDGAR splits → build the 100-label instruction → stratified subsample with `SEED=42` → build examples → sanity checks → save JSONL) is **byte-for-byte identical** to the fine-tuning notebook, which guarantees the validation set here is exactly the one used there. All artifacts are written to a **separate `no_finetune_baseline_task3/` subdirectory** so they never overwrite the fine-tuned run's outputs (or the Task 1/2 baselines').


In [ ]:
# --- Pin to a single GPU BEFORE torch is imported anywhere ---
# Kaggle "GPU T4 x2" exposes 2 GPUs. device_map="auto" then shards the model
# across cuda:0/cuda:1. At loss time TRL's _chunked_cross_entropy_loss builds the
# label mask on cuda:0 while the final hidden_states/lm_head live on cuda:1 ->
# "indices should be either on cpu or on the same device as the indexed tensor
# (cuda:1)". An 8B model in 4-bit (~5-6 GB) fits in ONE T4 (16 GB), so hide GPU 1.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- Kaggle: install the library versions this notebook expects (no-op locally) ---
# The Kaggle base image ships older trl/peft; pin trl 1.x so SFTConfig,
# completion_only_loss and processing_class are available.
if os.path.exists("/kaggle"):
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==4.55.4",   # MUST be <4.56: the 4.56 "core_model_loading"
                                              # threaded loader breaks bnb 4-bit -> full fp16 load -> OOM
                    "bitsandbytes==0.46.1",
                    "accelerate==1.7.0",
                    "peft==0.15.2",
                    "trl==0.20.0",
                    "datasets"], check=True)


In [ ]:
import sys; print("UTF-8 mode:", sys.flags.utf8_mode)

# Step 3.0 : Environment config + imports

Same pattern as the T1/T2 notebooks. Locally, LEDGAR lives at `data/LEDGAR/`; on Kaggle it mounts read-only and generated JSONL must be written to the writable `WORK_DIR` (`/kaggle/working`).

In [ ]:
# --- Environment config: run unchanged locally OR on Kaggle ---
import os
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()

if ON_KAGGLE:
    # Read-only mounted dataset. Must match the LEDGAR dataset-metadata id / mounted folder.
    DATA_DIR = Path("/kaggle/input/ledgar-lexglue")
    # Only this dir is writable AND persisted as kernel output:
    WORK_DIR = Path("/kaggle/working")
else:
    # LEDGAR sits in its own folder, sibling to CUAD_v1.
    DATA_DIR = Path(os.getenv("LEDGAR_DIR", str(Path(os.getenv("DATA_DIR", "data")) / "LEDGAR")))
    WORK_DIR = Path(".")

print(f"ON_KAGGLE={ON_KAGGLE} | DATA_DIR={DATA_DIR} | WORK_DIR={WORK_DIR}")

# All artifacts from THIS (no-fine-tune) notebook go under a dedicated subdirectory
# so they never overwrite the fine-tuned Task 3 run's outputs (eval_metrics.json,
# ledgar/, ...) nor the Task 1/2 baselines' directories.
OUTPUT_DIR = WORK_DIR / "no_finetune_baseline_task3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"OUTPUT_DIR={OUTPUT_DIR}")

In [ ]:
import json
import pandas as pd
from collections import Counter

# Reproducibility + sampling knobs (see TASK3_EDA_FINDINGS.md, Finding 1).
SEED = 42
TRAIN_PER_LABEL = 100   # ~100 x 100 labels -> ~10k train examples (rare labels cap lower)
VAL_PER_LABEL = 20      # ~20 x 100 labels  -> ~2k validation examples

pd.set_option("display.max_colwidth", 100)

# Step 3.1 : Load the LEDGAR splits + label map

`scripts/download_ledgar.py` wrote three flat CSVs (`text, label, label_name`) and a `labels.json` id→name map. We only need **train** and **validation** here (test is held out for the final exam).

In [ ]:
# The 100 allowed labels, in id order (canonical menu shown to the model).
with open(DATA_DIR / "labels.json", encoding="utf-8") as fh:
    id2label = {int(k): v for k, v in json.load(fh).items()}
LABELS = [id2label[i] for i in range(len(id2label))]
ALLOWED = set(LABELS)
print(f"{len(LABELS)} labels loaded. e.g. {LABELS[:3]} ... {LABELS[-3:]}")

train_full = pd.read_csv(DATA_DIR / "ledgar_train.csv")
val_full = pd.read_csv(DATA_DIR / "ledgar_validation.csv")
print(f"Full train: {len(train_full):>6} rows | Full validation: {len(val_full):>6} rows")
train_full.head(3)

# Step 3.2 : Build the instruction (100-label menu) + prompt template

The model can only pick a label it has been shown, so the instruction lists **all 100 labels**. This is the same string validated in the EDA notebook (~331 tokens) and is reused verbatim for training, validation, and the base-model baseline so every comparison is fair. The `### Instruction / ### Input / ### Response` template matches T1/T2.

In [ ]:
LABEL_LIST_STR = ", ".join(LABELS)
INSTRUCTION = (
    "Classify the following contract provision. "
    f"Answer with exactly one label from this list: [{LABEL_LIST_STR}]."
)
# Same template as T1/T2 (the completion during training = the label string).
PROMPT_TEMPLATE = "### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n"

print(f"Instruction length: {len(INSTRUCTION)} chars")
print(INSTRUCTION[:300] + " ...")

# Step 3.3 : Stratified subsample

Training on all 60k rows would drown the model in common labels (*Governing Laws* alone has 3,167). Instead we take ~`TRAIN_PER_LABEL` examples **per label** so every provision type is represented. Labels with fewer examples than the target (e.g. *Books* = 23, *Assigns* = 31) simply take all they have — that per-label floor is expected and macro-F1 will reflect it. We sample **within** LexGLUE's official splits, so there is no leakage to worry about.

In [ ]:
def stratified_sample(df, per_label, seed=SEED):
    """Take up to `per_label` rows for each label, then shuffle the result."""
    parts = []
    for name, group in df.groupby("label_name"):
        n = min(len(group), per_label)
        parts.append(group.sample(n=n, random_state=seed))
    out = pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    return out

train_sample = stratified_sample(train_full, TRAIN_PER_LABEL)
val_sample = stratified_sample(val_full, VAL_PER_LABEL)

print(f"train: {len(train_full)} -> {len(train_sample)} rows "
      f"({train_sample['label_name'].nunique()}/100 labels)")
print(f"val  : {len(val_full)} -> {len(val_sample)} rows "
      f"({val_sample['label_name'].nunique()}/100 labels)")

# Labels that hit the per-label floor (fewer examples than the target).
train_counts = train_sample["label_name"].value_counts()
capped = sorted(train_counts[train_counts < TRAIN_PER_LABEL].index)
print(f"\nLabels below the {TRAIN_PER_LABEL}/label target in train ({len(capped)}): "
      f"{[(c, int(train_counts[c])) for c in capped]}")

# Step 3.4 : Build the JSONL examples

Same schema as T1/T2 — `instruction`, `category`, `input`, `output` — where for T3 both `category` and `output` are the gold label name (kept as a field for per-label eval later).

In [ ]:
def build_examples(df):
    examples = []
    for row in df.itertuples(index=False):
        label = row.label_name
        examples.append({
            "instruction": INSTRUCTION,
            "category": label,        # gold label name (per-label eval hook)
            "input": str(row.text),
            "output": label,          # completion the model must generate
        })
    return examples

train_data = build_examples(train_sample)
val_data = build_examples(val_sample)
print(f"Built {len(train_data)} train and {len(val_data)} validation examples.")

# Show one full example end-to-end (prompt + expected completion).
ex = train_data[0]
print("\n--- Example ---")
print(PROMPT_TEMPLATE.format(instruction="<100-label instruction>", input=ex["input"]) + ex["output"])
print("\ncategory/output:", ex["category"])

# Step 3.5 : Sanity checks

Mirror T2's pre-save checks, adapted for classification:
1. **Every `output` is a valid label** — verbatim one of the 100 allowed labels (this is T3's equivalent of T2's "output parses as JSON").
2. **Label coverage** — every label appears in **train** (required). Validation coverage is *reported*, not required: the EDA showed `Books` is absent from the validation split (see TASK3_EDA_FINDINGS.md).
3. **Per-label counts** look sane (no empty inputs).

In [ ]:
def check(data, name, require_full_coverage):
    print(f"=== {name} ({len(data)} examples) ===")
    outputs = [e["output"] for e in data]

    # 1. every output is a valid, allowed label
    bad = [o for o in outputs if o not in ALLOWED]
    assert not bad, f"{len(bad)} outputs are not in the 100-label list! e.g. {bad[:5]}"
    print(f"  [OK] all {len(outputs)} outputs are valid labels")

    # 2. no empty / blank inputs
    empty = [e for e in data if not str(e["input"]).strip()]
    assert not empty, f"{len(empty)} examples have empty input text"
    print(f"  [OK] no empty inputs")

    # 3. label coverage
    present = set(outputs)
    missing = sorted(ALLOWED - present)
    print(f"  coverage: {len(present)}/100 labels present")
    if missing:
        print(f"  {'[FAIL]' if require_full_coverage else '[warn]'} missing labels: {missing}")
    if require_full_coverage:
        assert not missing, f"{name} is missing {len(missing)} labels: {missing}"

    # 4. per-label count summary
    counts = Counter(outputs)
    print(f"  per-label counts: min={min(counts.values())} "
          f"max={max(counts.values())} mean={sum(counts.values())/len(counts):.1f}")
    print()

check(train_data, "train", require_full_coverage=True)
check(val_data, "validation", require_full_coverage=False)
print("Sanity checks passed.")

# Step 3.6 : Save to JSONL

Write the generated JSONL under **`OUTPUT_DIR`** (`no_finetune_baseline_task3/`). Same `SEED=42` stratified sample as the fine-tuned run, so the validation JSONL saved here is identical to the one the fine-tuned notebook evaluates on — the comparison notebook verifies that record-by-record.

In [ ]:
def save_jsonl(data, filename):
    with open(filename, "w", encoding="utf-8") as f:
        for entry in data:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

# Under OUTPUT_DIR so the baseline's artifacts never collide with the fine-tuned run's.
LEDGAR_TRAIN_PATH = OUTPUT_DIR / "ledgar" / "train"
LEDGAR_VALIDATION_PATH = OUTPUT_DIR / "ledgar" / "validation"
LEDGAR_TRAIN_PATH.mkdir(parents=True, exist_ok=True)
LEDGAR_VALIDATION_PATH.mkdir(parents=True, exist_ok=True)

train_file = LEDGAR_TRAIN_PATH / "ledgar_task3_train.jsonl"
val_file = LEDGAR_VALIDATION_PATH / "ledgar_task3_validation.jsonl"
save_jsonl(train_data, train_file)
save_jsonl(val_data, val_file)
print(f"Saved {len(train_data)} train -> {train_file}")
print(f"Saved {len(val_data)} validation -> {val_file}")

# Load the base model for inference (no training)

Instead of fine-tuning, load **`meta-llama/Meta-Llama-3.1-8B`** in 4-bit NF4 — exactly how the fine-tuning notebook loads its *base* model — then use it directly for generation. There is **no `SFTTrainer`, no `LoraConfig`, no completion-only loss, and no adapter saved to disk**. We only need the frozen base model to produce label predictions on the validation set built above.

- Note : before execution of the cell below run to the terminal `$env:HF_TOKEN=your_hf_token`

In [ ]:
# Diagnostic: confirm WHICH account the token belongs to and whether it can access the gated repo.
# A 403 "not in the authorized list" means the token is valid but this account lacks access.
import os
from huggingface_hub import whoami, auth_check
from huggingface_hub.utils import GatedRepoError, HfHubHTTPError

def get_hf_token():
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    from dotenv import load_dotenv
    load_dotenv()
    return os.getenv("HF_TOKEN")

hf_token = get_hf_token()
assert hf_token, "HF_TOKEN not found (Kaggle Secret or local .env)"

# Base model for the real (non-smoke-test) run. 8B in 4-bit NF4 is ~5-6 GB of weights,
# which fits fully in a single T4 (16 GB) VRAM with no CPU/disk offload. Smoke-test was "meta-llama/Llama-3.2-1B".
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B"

# 1) Which account is this token? Request access on the model page with THIS exact account.
me = whoami(token=hf_token)
print(f"Token belongs to: {me['name']}  (type: {me.get('type')})")

# 2) Does that account actually have access to the gated repo?
try:
    auth_check(MODEL_ID, token=hf_token)
    print(f"\u2705 Access granted to {MODEL_ID} — you can run the load cell below.")
except GatedRepoError:
    print(f"\u274c Still gated for account '{me['name']}'.")
    print(f"   -> Visit https://huggingface.co/{MODEL_ID} while logged in as '{me['name']}', "
          f"accept the license, and wait for approval.")
    print(f"   -> Or use the ungated mirror: model_name = 'unsloth/Meta-Llama-3.1-8B'")
except HfHubHTTPError as e:
    print(f"\u274c Auth/HTTP error (likely an invalid or expired token): {e}")


In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
import os

def get_hf_token():
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    from dotenv import load_dotenv
    load_dotenv()  # reads .env from the current working dir (project root)
    return os.getenv("HF_TOKEN")

hf_token = get_hf_token()
assert hf_token, "HF_TOKEN not found (Kaggle Secret or local .env)"

from huggingface_hub import login
login(token=hf_token)

# 1. Configuration - base model, loaded AS-IS (NO fine-tuning, NO adapter).
# 8B in 4-bit NF4 is ~5-6 GB of weights, which fits fully in a single T4 (16 GB).
model_name = "meta-llama/Meta-Llama-3.1-8B"
MAX_SEQ_LENGTH = 1024   # matches the fine-tuned Task 3 run

# 2. QLoRA-style 4-bit load (same quantization the fine-tuned run used for its base model)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# 3. Load Base Model (whole model on GPU 0; GPU 1 hidden in cell 1)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
    token=hf_token
)

# 4. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Loaded base model '{model_name}' in 4-bit for inference (no training).")

# Evaluate the base model on validation — valid-label rate, accuracy, macro/micro-F1, per-label, confusions

**The evaluation code below is the exact same code as the fine-tune notebook's eval cell** — same greedy generation (`do_sample=False`, `max_new_tokens=16`), same `predict_label` parsing (strip + lowercase, first line only, mapped to a canonical label or the `__INVALID__` sentinel), same metric order and aggregation. Only the model differs: here it is the **un-fine-tuned base model**, so these are the **baseline** numbers.

Comparing them against the fine-tuned notebook's metrics on this identical validation set shows exactly what fine-tuning bought, metric by metric:

1. **Valid-label rate** — did the base model answer with a bare label from the menu at all? (Expected to be the largest delta — base models chat, explain, and echo the prompt.)
2. **Accuracy** — fraction where predicted label == gold label.
3. **Macro-F1** — the headline number; LEDGAR is 137× imbalanced (EDA Finding 1).
4. **Micro-F1** — for LexGLUE-leaderboard comparison.

Metrics use `labels=LABELS` so invalid predictions count against recall without inflating any real label's precision. Results are written to the **`no_finetune_baseline_task3/` subdirectory** (`eval_metrics.json`, `eval_report.txt`) so they sit alongside — but never overwrite — the fine-tuned run's artifacts. The only schema addition over the fine-tuned file is `"fine_tuned": false`.

In [ ]:
from sklearn.metrics import f1_score, precision_recall_fscore_support

# Use cache for faster generation at inference time.
model.config.use_cache = True
model.eval()

# Canonical-label lookup for parsing (metric 1): strip + lowercase, first line only.
LABEL_BY_LOWER = {l.lower(): l for l in LABELS}
INVALID = "__INVALID__"   # sentinel for un-parseable predictions (not in LABELS)

def predict_label(example):
    """Greedy-generate and parse the completion into (canonical_label, is_valid)."""
    prompt = (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Input:\n{example['input']}\n\n"
        f"### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=MAX_SEQ_LENGTH).to(model.device)
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        out = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()
    text = raw.split("\n", 1)[0].strip()   # first line only
    canonical = LABEL_BY_LOWER.get(text.lower())
    return (canonical, True) if canonical is not None else (INVALID, False)

gold, pred, valid_flags = [], [], []
for i, ex in enumerate(val_data):
    label, is_valid = predict_label(ex)
    gold.append(ex["category"])
    pred.append(label)
    valid_flags.append(is_valid)
    if (i + 1) % 100 == 0:
        print(f"  evaluated {i + 1}/{len(val_data)}")

# --- Aggregate metrics (labels=LABELS: invalid preds count as misses, never FPs) ---
n = len(gold)
valid_label_rate = sum(valid_flags) / n
accuracy = sum(g == p for g, p in zip(gold, pred)) / n
macro_f1 = f1_score(gold, pred, labels=LABELS, average="macro", zero_division=0)
micro_f1 = f1_score(gold, pred, labels=LABELS, average="micro", zero_division=0)
prec, rec, f1, support = precision_recall_fscore_support(
    gold, pred, labels=LABELS, zero_division=0)

# Top confused (gold -> pred) pairs, most frequent first.
conf = Counter((g, p) for g, p in zip(gold, pred) if g != p)
top_conf = conf.most_common(15)

print(f"\nValidation examples : {n}")
print(f"Valid-label rate    : {valid_label_rate:.4f}")
print(f"Accuracy            : {accuracy:.4f}")
print(f"Macro-F1 (headline) : {macro_f1:.4f}")
print(f"Micro-F1            : {micro_f1:.4f}")

print("\nTop 15 confused (gold -> pred) pairs:")
for (g, p), c in top_conf:
    print(f"  {c:3d}  {g}  ->  {p}")

# --- Per-label table (sorted by F1 ascending: worst offenders first) ---
per_label_rows = sorted(
    ({"label": LABELS[i], "precision": prec[i], "recall": rec[i],
      "f1": f1[i], "support": int(support[i])} for i in range(len(LABELS))),
    key=lambda r: r["f1"],
)
hdr = f"{'Label':40s} {'precision':>9s} {'recall':>7s} {'f1':>6s} {'support':>8s}"
tbl = [hdr, "-" * len(hdr)]
for r in per_label_rows:
    tbl.append(f"{r['label']:40s} {r['precision']:9.2f} {r['recall']:7.2f} "
               f"{r['f1']:6.2f} {r['support']:8d}")
per_label_table = "\n".join(tbl)

# --- Persist artifacts (same filenames/convention as T1/T2; +fine_tuned flag) ---
eval_metrics = {
    "model_name": model_name,
    "fine_tuned": False,
    "task": "task3_provision_classification",
    "n_validation_examples": n,
    "valid_label_rate": float(valid_label_rate),
    "accuracy": float(accuracy),
    "macro_f1": float(macro_f1),
    "micro_f1": float(micro_f1),
    "per_label": {
        LABELS[i]: {"precision": float(prec[i]), "recall": float(rec[i]),
                    "f1": float(f1[i]), "support": int(support[i])}
        for i in range(len(LABELS))
    },
    "top_confusions": [
        {"gold": g, "pred": p, "count": c} for (g, p), c in top_conf
    ],
}
eval_metrics_path = OUTPUT_DIR / "eval_metrics.json"
with open(eval_metrics_path, "w", encoding="utf-8") as f:
    json.dump(eval_metrics, f, indent=2)

eval_report_path = OUTPUT_DIR / "eval_report.txt"
with open(eval_report_path, "w", encoding="utf-8") as f:
    f.write("Task 3 — LEDGAR provision classification (validation, BASELINE, no fine-tuning)\n\n")
    f.write(f"Validation examples : {n}\n")
    f.write(f"Valid-label rate    : {valid_label_rate:.4f}\n")
    f.write(f"Accuracy            : {accuracy:.4f}\n")
    f.write(f"Macro-F1 (headline) : {macro_f1:.4f}\n")
    f.write(f"Micro-F1            : {micro_f1:.4f}\n\n")
    f.write("Top 15 confused (gold -> pred) pairs:\n")
    for (g, p), c in top_conf:
        f.write(f"  {c:3d}  {g}  ->  {p}\n")
    f.write("\nPer-label precision / recall / F1 (worst F1 first):\n")
    f.write(per_label_table + "\n")

print(f"\nWrote eval metrics to {eval_metrics_path}")
print(f"Wrote eval report  to {eval_report_path}")